# Final Product of Our Project!

In [ ]:
from ultralytics import YOLO
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import torch
import cv2
import os
import sys
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from transformers import pipeline
from transformers.utils.quantization_config import BitsAndBytesConfig
import torch
import gradio as gr
from langchain_community.llms import HuggingFacePipeline
from langchain.agents import initialize_agent, Tool
%matplotlib inline
%load_ext autoreload
%autoreload 2

/home/stellarlane/miniconda3/envs/polyp/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("Loading YOLO...", end = "")
yolo = YOLO('../../yolo/models/yolo11n-trained-on-mixed-cli-col-ETIS.pt')
print("Completed \nLoading SAM2...", end = "")

sam2_checkpoint = "../checkpoints/sam2_hiera_tiny.pt"
model_cfg = "configs/sam2/sam2_hiera_t.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device="cpu") 
sam2_model = torch.load("./fined_tuned_1.pt", weights_only=False)
sam2_predictor = SAM2ImagePredictor(sam2_model, device="cpu")
print("Completed \nLoading MedGemma...", end = "")

model_id = f"google/medgemma-4b-it"
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", llm_int8_enable_fp32_cpu_offload=True)
model_kwargs = dict(torch_dtype=torch.bfloat16, device_map="auto", quantization_config=quantization_config)
medgemma_pipe = pipeline("image-text-to-text", model=model_id, model_kwargs=model_kwargs)
medgemma_pipe.model.generation_config.do_sample = False
print("All loading Completed.")

Loading YOLO...Completed 
Loading SAM2...Completed 
Loading MedGemma...Completed 
Loading MedGemma...

Loading checkpoint shards: 100%|██████████| 2/2 [01:23<00:00, 41.83s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0
Device set to use cuda:0


All loading Completed.


In [24]:
def yolo_sam(image):
    results = yolo.predict(image)
    boxes = results[0].boxes.xyxy[0].cpu().numpy()
    sam2_predictor.set_image(image)  
    masks, _, _ = sam2_predictor.predict(
        box = boxes,
        multimask_output=True,
    )
    mask_image = Image.fromarray((masks[0] * 255).astype(np.uint8))
    plt.imshow(Image.open("./1.png"))
    plt.imshow(mask_image, cmap='coolwarm', alpha=0.5)
    plt.axis('off')
    plt.savefig("results.png", bbox_inches='tight', pad_inches=0)
    return Image.open("results.png")

In [25]:
def describe_img(image):
    prompt = "Describe this polyp image, the area of polyp is the red parts of the second picture, describe its potential effects and ways to treat it, try to give specific suggestions based on the picture, output in Chinese"
    system_instruction = "You are an expert polyp segmenter and analyst."
    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": "system_instruction"}]
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image", "image": image},
                {"type": "image", "image": Image.open("./results.png")}
            ]
        }
    ]
    output = medgemma_pipe(text=messages, max_new_tokens=512)
    return output[0]['generated_text'][-1]['content']

In [ ]:
def process_image(input_image):
    # Dummy processing: just return the input image twice and a sample text
    # Replace with your actual processing logic
    output_image1 = yolo_sam(input_image)
    output_text = describe_img(input_image)
    return output_image1, output_text


css = """

"""

iface = gr.Interface(
    fn=process_image,
    inputs=gr.Image(type="pil", label="Input image"),
    outputs=[
        gr.Image(type="pil", label="Segmentation Results"),
        gr.Markdown(label="Analysis from MedGemma")
    ],
    title="",
    css=css
)



iface.launch()


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.



0: 480x640 1 target, 64.2ms
0: 480x640 1 target, 64.2ms
Speed: 3.4ms preprocess, 64.2ms inference, 4.8ms postprocess per image at shape (1, 3, 480, 640)
Speed: 3.4ms preprocess, 64.2ms inference, 4.8ms postprocess per image at shape (1, 3, 480, 640)


/home/stellarlane/miniconda3/envs/polyp/lib/python3.13/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/stellarlane/miniconda3/envs/polyp/lib/python3.13/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `64` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
